# Label the frame pairs — four quadrants + a binary gate decision

Reads `feature_dataset_all.csv` from Drive, picks a threshold for each of the two teacher
scores, and writes back a labelled copy.

## What gets added

| New column | Values |
|---|---|
| `quad` | `0..3` — bitmask, `2*frame_high + gaze_high` |
| `quad_label` | `TRANSITION` / `PURSUIT` / `REFIXATION` / `STABLE` |
| `gate` | `SEND` / `DISCARD` |
| `frame_high`, `gaze_high` | the two bits, kept separately for convenience |
| `split` | `train` / `val` / `test`, assigned **by video** |

## The 2 × 2

| | gaze HIGH | gaze LOW |
|---|---|---|
| **frame HIGH** | `STABLE` (3) — **DISCARD** | `REFIXATION` (2) — SEND |
| **frame LOW** | `PURSUIT` (1) — SEND | `TRANSITION` (0) — SEND |

## On file size

17.9 MB looks big but is ~13,000 rows; pandas handles it without ceremony. The bytes are
almost all in the packed string columns — `gaze_rates_window`, `imu_window`,
`gaze_xy_window`, `gaze_vec3d_rates_window` — which thresholding never touches.

So this notebook reads **3 columns** for the analysis (about 5% of the file) and only
loads the whole thing once, at the end, to write the labels. No dask, no chunking, no
parquet — those would be machinery for a problem you do not have.

## Order of operations

Thresholds are chosen **after** the diagnostics, not before: whether the median is the
right cut depends on the `REFIXATION` share and on how much of the label is really scene
identity. Both are measured in section 3.

## 1 — Mount Drive and load the two columns that matter

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

DRIVE_DIR = "/content/drive/MyDrive/GazeVLM"
CSV_IN    = os.path.join(DRIVE_DIR, "feature_dataset_all.csv")
CSV_OUT   = os.path.join(DRIVE_DIR, "feature_dataset_all_labeled.csv")

assert os.path.exists(CSV_IN), f"not found: {CSV_IN}"
print(f"input : {CSV_IN}  ({os.path.getsize(CSV_IN)/1e6:.1f} MB)")
print(f"output: {CSV_OUT}")

# Only what thresholding needs. The packed string columns are >95% of the bytes.
NEED = ["sequence", "frame_similarity", "gaze_patch_token_sim"]

t0 = time.time()
lite = pd.read_csv(CSV_IN, usecols=NEED)
t_lite = time.time() - t0

print(f"\nread {len(lite):,} rows x {len(NEED)} columns in {t_lite:.1f}s")
print(f"   videos          : {lite['sequence'].nunique()}")
print(f"   in memory       : {lite.memory_usage(deep=True).sum()/1e6:.1f} MB")
print(f"   vs file on disk : {os.path.getsize(CSV_IN)/1e6:.1f} MB"
      f"   ({100*lite.memory_usage(deep=True).sum()/os.path.getsize(CSV_IN):.0f}%)")

assert lite[NEED[1:]].notna().all().all(), "NaN in the score columns"
print(f"\n   frame_similarity     mean {lite.frame_similarity.mean():.3f}  "
      f"std {lite.frame_similarity.std():.3f}  "
      f"range [{lite.frame_similarity.min():.3f}, {lite.frame_similarity.max():.3f}]")
print(f"   gaze_patch_token_sim mean {lite.gaze_patch_token_sim.mean():.3f}  "
      f"std {lite.gaze_patch_token_sim.std():.3f}  "
      f"range [{lite.gaze_patch_token_sim.min():.3f}, {lite.gaze_patch_token_sim.max():.3f}]")
print("\n   The two spreads differ, which is why they need SEPARATE thresholds.")

## 2 — Split by video, first

Thresholds are computed on the **training split only** — percentiles taken over the whole
file leak validation and test information into the labels themselves.

The split must be **by video**: consecutive rows are one second apart and near-duplicates,
so a random row split would put almost-identical footage on both sides.

Videos are sorted by mean `frame_similarity` and dealt round-robin, so each split spans
the full range of scene types rather than getting a clump of static or busy videos.

In [ ]:
TRAIN, VAL, TEST = 7, 2, 1          # out of every 10 videos

vm = lite.groupby("sequence")["frame_similarity"].mean().sort_values()
split_of = {}
for i, seq in enumerate(vm.index):
    r = i % (TRAIN + VAL + TEST)
    split_of[seq] = "train" if r < TRAIN else ("val" if r < TRAIN + VAL else "test")

lite["split"] = lite["sequence"].map(split_of)

print("videos per split:")
vs = pd.Series(split_of).value_counts()
for s in ("train", "val", "test"):
    rows = int((lite.split == s).sum())
    print(f"   {s:5s} {vs.get(s,0):3d} videos   {rows:6,} rows   ({100*rows/len(lite):4.1f}%)")

print("\nmean frame_similarity per split (should be close -- that is the point of "
      "dealing sorted videos):")
print(lite.groupby("split")[["frame_similarity","gaze_patch_token_sim"]].mean().round(4))

train = lite[lite.split == "train"]
print(f"\nthresholds will be computed on {len(train):,} training rows from "
      f"{train['sequence'].nunique()} videos")

## 3 — Diagnostics: is the median the right cut?

Three questions, in order of how much they change the answer.

1. **Is either distribution bimodal?** If yes, a data-driven cut (Otsu / GMM) is
   principled. If unimodal — the usual case for cosine similarity — the median is honest
   and anything fancier is decoration around the same number.
2. **How much of the score is scene identity?** The intraclass correlation is
   between-video variance ÷ total variance. Above ~0.4 a global threshold largely encodes
   *which video this is*, and since the split is by video, train and val would end up with
   different class balances.
3. **Does the joint distribution leave `REFIXATION` usable?** Median cuts guarantee 50/50
   on each axis and nothing about the four cells.

In [ ]:
def icc1(x, g):
    # One-way ICC: between-video variance as a share of the total, corrected for
    # unequal group sizes. 0 = video identity says nothing, 1 = it says everything.
    d = pd.DataFrame({"x": np.asarray(x, float), "g": np.asarray(g)})
    k, N = d.g.nunique(), len(d)
    if k < 2:
        return float("nan")
    gm = d.groupby("g")["x"].agg(["mean", "count"])
    grand = d.x.mean()
    msb = (gm["count"] * (gm["mean"] - grand) ** 2).sum() / (k - 1)
    ssw = sum(((sub.x - sub.x.mean()) ** 2).sum() for _, sub in d.groupby("g"))
    msw = ssw / (N - k)
    n0 = (N - (gm["count"] ** 2).sum() / N) / (k - 1)
    return float((msb - msw) / (msb + (n0 - 1) * msw))


def bimodal_bic(x):
    # 2-component GMM vs 1. Negative delta => two components explain the data better,
    # i.e. there really are two humps and a data-driven cut is meaningful.
    from sklearn.mixture import GaussianMixture
    v = np.asarray(x, float).reshape(-1, 1)
    b1 = GaussianMixture(1, random_state=0).fit(v).bic(v)
    b2 = GaussianMixture(2, random_state=0).fit(v).bic(v)
    return b2 - b1


print("=" * 74)
for c in ("frame_similarity", "gaze_patch_token_sim"):
    icc = icc1(train[c], train["sequence"])
    dbic = bimodal_bic(train[c])
    print(f"{c}")
    print(f"   ICC (scene identity share) : {icc:+.3f}   "
          f"{'<- HIGH, see note below' if icc > 0.4 else 'ok'}")
    print(f"   BIC(2) - BIC(1)            : {dbic:+,.0f}   "
          f"{'bimodal -- a data-driven cut is meaningful' if dbic < 0 else 'unimodal -- use the median'}")
    print(f"   median                     : {train[c].median():.4f}")
print("=" * 74)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
for a, c, col in zip(ax[:2], ("frame_similarity", "gaze_patch_token_sim"),
                     ("tab:blue", "tab:orange")):
    a.hist(train[c], bins=60, color=col, alpha=.8)
    a.axvline(train[c].median(), c="k", ls="--", lw=1.2, label=f"median {train[c].median():.3f}")
    a.set_xlabel(c); a.set_ylabel("rows"); a.legend()
    a.set_title(f"{c}\n(one hump = median; two = consider GMM)")

ax[2].scatter(lite.frame_similarity, lite.gaze_patch_token_sim, s=2, alpha=.08)
ax[2].axvline(train.frame_similarity.median(), c="k", lw=.9, ls="--")
ax[2].axhline(train.gaze_patch_token_sim.median(), c="k", lw=.9, ls="--")
ax[2].set_xlabel("frame_similarity"); ax[2].set_ylabel("gaze_patch_token_sim")
r = np.corrcoef(lite.frame_similarity, lite.gaze_patch_token_sim)[0,1]
ax[2].set_title(f"joint distribution — r = {r:+.3f}\n(positive r starves the off-diagonal cells)")
plt.tight_layout(); plt.show()

print(f"correlation between the two scores: r = {r:+.3f}")
print("   The more positive this is, the emptier REFIXATION and PURSUIT will be,")
print("   because the scores tend to be high or low together.")

### Sensitivity: how the four cells respond to the cut

Never commit to one threshold pair without seeing the neighbours. If the class shares
swing wildly across nearby percentiles, the labelling is fragile and any downstream result
is really a statement about the cut.

In [ ]:
NAMES = {0: "TRANSITION", 1: "PURSUIT", 2: "REFIXATION", 3: "STABLE"}

def quad_of(fs, gs, tf, tg):
    return 2 * (np.asarray(fs) > tf).astype(int) + (np.asarray(gs) > tg).astype(int)

def shares(tf, tg, d=None):
    d = lite if d is None else d
    q = quad_of(d.frame_similarity, d.gaze_patch_token_sim, tf, tg)
    return {NAMES[i]: 100 * float((q == i).mean()) for i in range(4)}

grid = [(40,40),(40,50),(40,60),(50,40),(50,50),(50,60),(60,40),(60,50),(60,60)]
rows = []
for fp, gp in grid:
    tf = np.percentile(train.frame_similarity, fp)
    tg = np.percentile(train.gaze_patch_token_sim, gp)
    s = shares(tf, tg)
    rows.append(dict(frame_pct=fp, gaze_pct=gp, tau_frame=round(tf,4), tau_gaze=round(tg,4),
                     **{k: round(v,1) for k,v in s.items()},
                     min_cell=round(min(s.values()),1)))
sens = pd.DataFrame(rows)
display(sens)

best = sens.loc[sens.min_cell.idxmax()]
print(f"most balanced pair on this grid: frame_pct {best.frame_pct:.0f} / "
      f"gaze_pct {best.gaze_pct:.0f}  ->  smallest cell {best.min_cell:.1f}%")
print("   (balance is one goal among several -- see the note under section 4)")

## 4 — Choose the thresholds

Defaults are the medians, computed on train. Two reasons to move off them, both visible
above:

- **`REFIXATION` under ~5%** — the class that justifies having two thresholds at all. To
  grow it: **lower** `FRAME_PCT`, **raise** `GAZE_PCT`.
- **High ICC** — set `PER_VIDEO = True` to take percentiles within each video instead.
  This removes the scene confound, but it changes what the label *means*, from "dissimilar"
  to "more dissimilar than usual **for this scene**" — which is not what a fixed-threshold
  gate does at deployment. Use it knowingly, and say so in the write-up.

`DEAD_ZONE` optionally marks rows within a percentile band of either cut as `AMBIGUOUS`.
Those rows are noise-dominated — a 0.001 cosine difference decides their class — so
excluding them buys cleaner labels at the cost of some data.

In [ ]:
FRAME_PCT = 50        # percentile for tau_frame, on the TRAIN split
GAZE_PCT  = 50        # percentile for tau_gaze
PER_VIDEO = False     # True -> percentiles within each video (see the note above)
DEAD_ZONE = 0.0       # e.g. 5.0 marks rows within +/-5 percentile of a cut as AMBIGUOUS
MIN_CELL  = 5.0       # warn if any quadrant falls below this share (%)

if not PER_VIDEO:
    TAU_F = float(np.percentile(train.frame_similarity, FRAME_PCT))
    TAU_G = float(np.percentile(train.gaze_patch_token_sim, GAZE_PCT))
    print(f"GLOBAL thresholds (from {len(train):,} train rows)")
    print(f"   tau_frame = {TAU_F:.4f}   (p{FRAME_PCT} of frame_similarity)")
    print(f"   tau_gaze  = {TAU_G:.4f}   (p{GAZE_PCT} of gaze_patch_token_sim)")
    tf_row = np.full(len(lite), TAU_F)
    tg_row = np.full(len(lite), TAU_G)
else:
    # per-video percentiles, still computed from train rows only where a video has them
    src = train if (train.sequence.nunique() == lite.sequence.nunique()) else lite
    tf_map = src.groupby("sequence").frame_similarity.quantile(FRAME_PCT/100)
    tg_map = src.groupby("sequence").gaze_patch_token_sim.quantile(GAZE_PCT/100)
    tf_row = lite.sequence.map(tf_map).to_numpy()
    tg_row = lite.sequence.map(tg_map).to_numpy()
    TAU_F, TAU_G = float(np.median(tf_row)), float(np.median(tg_row))
    print(f"PER-VIDEO thresholds: {len(tf_map)} pairs, median "
          f"tau_frame {TAU_F:.4f} / tau_gaze {TAU_G:.4f}")
    print("   NOTE: the label now means 'relative to this scene', not an absolute level.")

fs = lite.frame_similarity.to_numpy()
gs = lite.gaze_patch_token_sim.to_numpy()
quad = 2 * (fs > tf_row).astype(int) + (gs > tg_row).astype(int)
lite["quad"] = quad

counts = pd.Series(quad).value_counts().reindex(range(4), fill_value=0)
print("\n" + "=" * 62)
print(f"{'code':>4}  {'label':<12} {'rows':>8} {'share':>8}   gate")
print("-" * 62)
for i in range(4):
    print(f"{i:>4}  {NAMES[i]:<12} {counts[i]:>8,} {100*counts[i]/len(lite):>7.1f}%   "
          f"{'DISCARD' if i == 3 else 'SEND'}")
print("=" * 62)

low = [NAMES[i] for i in range(4) if 100*counts[i]/len(lite) < MIN_CELL]
if low:
    print(f"\n!! below {MIN_CELL}%: {low}")
    if "REFIXATION" in low:
        print("   REFIXATION is the case that justifies two thresholds instead of one.")
        print(f"   To grow it: LOWER FRAME_PCT (from {FRAME_PCT}) and RAISE GAZE_PCT "
              f"(from {GAZE_PCT}), then re-run this cell.")
    if "PURSUIT" in low:
        print("   PURSUIT is suppressed by the coarse 7x7 patch grid -- one cell covers")
        print("   ~25 deg of FOV, so the 'attended region' carries a lot of background.")
        print("   A small count here is expected and is NOT evidence pursuit is absent.")
else:
    print(f"\nall four quadrants above {MIN_CELL}%")

print("\n2 x 2 (rows = frame, cols = gaze):")
m = pd.DataFrame([[counts[3], counts[2]], [counts[1], counts[0]]],
                 index=["frame HIGH", "frame LOW"], columns=["gaze HIGH", "gaze LOW"])
display(m)

### Does the balance survive within each video?

Global shares can look perfect while individual videos are 95/5 — that is the scene-identity
problem, and it is invisible in the totals.

In [ ]:
pv = (lite.groupby("sequence")["quad"]
      .value_counts(normalize=True).unstack(fill_value=0) * 100)
pv = pv.reindex(columns=range(4), fill_value=0).rename(columns=NAMES).round(1)
pv["dominant"] = pv.idxmax(axis=1)
pv["max_share"] = pv[[NAMES[i] for i in range(4)]].max(axis=1).round(1)
display(pv.sort_values("max_share", ascending=False).head(12))

skewed = pv[pv.max_share > 80]
print(f"videos where one class holds >80% of rows: {len(skewed)} of {len(pv)}")
if len(skewed):
    print("   These are scenes the threshold cannot discriminate within. If most videos")
    print("   look like this, the label is tracking scene identity -- set PER_VIDEO = True.")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
pv[[NAMES[i] for i in range(4)]].plot(kind="hist", bins=25, alpha=.55, ax=ax[0])
ax[0].set_xlabel("share of a video's rows (%)"); ax[0].set_title("per-video class share")
sp = (lite.groupby("split")["quad"].value_counts(normalize=True).unstack(fill_value=0)*100)
sp.reindex(columns=range(4), fill_value=0).rename(columns=NAMES).plot(kind="bar", ax=ax[1])
ax[1].set_ylabel("% of rows"); ax[1].set_title("class balance across splits")
ax[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

print("\nclass balance per split (train/val/test should look alike):")
display(sp.reindex(columns=range(4), fill_value=0).rename(columns=NAMES).round(1))

## 5 — Write the labels onto the full CSV

Now — and only now — the whole file is read. The label columns are computed from `lite`,
which is row-aligned with it, so nothing is recomputed and nothing can drift.

Written to a **new** file; the input is left untouched.

In [ ]:
t0 = time.time()
full = pd.read_csv(CSV_IN)
print(f"read {len(full):,} rows x {full.shape[1]} columns in {time.time()-t0:.1f}s "
      f"({full.memory_usage(deep=True).sum()/1e6:.0f} MB in memory)")

assert len(full) == len(lite), "row count changed between the two reads"
assert (full["sequence"].to_numpy() == lite["sequence"].to_numpy()).all(), "row order differs"
assert np.allclose(full["frame_similarity"], lite["frame_similarity"]), "score mismatch"
print("   [PASS] the two reads are row-aligned")

full["frame_high"] = (lite["quad"].to_numpy() >> 1).astype(int)
full["gaze_high"]  = (lite["quad"].to_numpy() & 1).astype(int)
full["quad"]       = lite["quad"].to_numpy()
full["quad_label"] = full["quad"].map(NAMES)
full["gate"]       = np.where(full["quad"] == 3, "DISCARD", "SEND")
full["split"]      = lite["split"].to_numpy()

if DEAD_ZONE > 0:
    # rows sitting within +/-DEAD_ZONE percentile of either cut are noise-dominated
    lo_f, hi_f = (np.percentile(train.frame_similarity, FRAME_PCT - DEAD_ZONE),
                  np.percentile(train.frame_similarity, FRAME_PCT + DEAD_ZONE))
    lo_g, hi_g = (np.percentile(train.gaze_patch_token_sim, GAZE_PCT - DEAD_ZONE),
                  np.percentile(train.gaze_patch_token_sim, GAZE_PCT + DEAD_ZONE))
    amb = (full.frame_similarity.between(lo_f, hi_f) |
           full.gaze_patch_token_sim.between(lo_g, hi_g))
    full.loc[amb, "quad_label"] = "AMBIGUOUS"
    full.loc[amb, "gate"] = "AMBIGUOUS"
    print(f"   DEAD_ZONE {DEAD_ZONE}%: {int(amb.sum()):,} rows marked AMBIGUOUS "
          f"({100*amb.mean():.1f}%)")

full.attrs = {}
full.to_csv(CSV_OUT, index=False)
print(f"\nwrote {len(full):,} rows x {full.shape[1]} columns -> {CSV_OUT}")
print(f"   {os.path.getsize(CSV_OUT)/1e6:.1f} MB  "
      f"(input was {os.path.getsize(CSV_IN)/1e6:.1f} MB, unchanged)")

json.dump(dict(tau_frame=TAU_F, tau_gaze=TAU_G, frame_pct=FRAME_PCT, gaze_pct=GAZE_PCT,
               per_video=PER_VIDEO, dead_zone=DEAD_ZONE,
               computed_on="train split", n_train_rows=int(len(train)),
               names=NAMES, source=os.path.basename(CSV_IN)),
          open(os.path.join(DRIVE_DIR, "thresholds.json"), "w"), indent=2)
print(f"   thresholds saved -> {DRIVE_DIR}/thresholds.json")

## 6 — Verify the saved file

In [ ]:
chk = pd.read_csv(CSV_OUT, usecols=["sequence","frame_similarity","gaze_patch_token_sim",
                                    "quad","quad_label","gate","frame_high","gaze_high","split"])

ok = [
    ("row count unchanged", len(chk) == len(lite)),
    ("quad in 0..3", chk["quad"].between(0, 3).all()),
    ("quad_label matches quad",
     bool((chk[chk.quad_label != "AMBIGUOUS"].quad.map(NAMES)
           == chk[chk.quad_label != "AMBIGUOUS"].quad_label).all())),
    ("gate == DISCARD iff quad == 3",
     bool(((chk.quad == 3) == (chk.gate == "DISCARD")).all()) if DEAD_ZONE == 0 else True),
    # numpy, not pandas: Series does not implement >> or &
    ("frame_high == quad >> 1",
     bool((chk.frame_high.to_numpy() == (chk["quad"].to_numpy() >> 1)).all())),
    ("gaze_high  == quad & 1",
     bool((chk.gaze_high.to_numpy() == (chk["quad"].to_numpy() & 1)).all())),
    ("bits agree with the raw scores",
     bool((chk.frame_high.astype(bool) == (chk.frame_similarity > TAU_F)).all())
     if not PER_VIDEO else True),
    ("every video is in exactly one split",
     bool((chk.groupby("sequence")["split"].nunique() == 1).all())),
    ("no NaN in the new columns",
     bool(chk[["quad","quad_label","gate","split"]].notna().all().all())),
]
for n, v in ok:
    print(f"   [{'PASS' if v else 'FAIL'}]  {n}")
print("\n" + ("ALL PASS" if all(v for _, v in ok) else "!!! CHECK THE FAILURES ABOVE"))

print("\nfinal label distribution:")
display(chk["quad_label"].value_counts().rename("rows").to_frame()
        .assign(pct=lambda d: (100*d.rows/len(chk)).round(1)))
print("gate decision:")
display(chk["gate"].value_counts().rename("rows").to_frame()
        .assign(pct=lambda d: (100*d.rows/len(chk)).round(1)))

keep = 100 * (chk.gate == "SEND").mean()
print(f"\nthe gate would send {keep:.1f}% of frames and skip {100-keep:.1f}%.")
print("   That skip rate is the compute saving the oracle could achieve; a learned")
print("   gate can only approach it, never beat it.")

display(chk.head(8)[["sequence","frame_similarity","gaze_patch_token_sim",
                     "frame_high","gaze_high","quad","quad_label","gate","split"]])

---

## What to do with this

```python
CSV = "/content/drive/MyDrive/GazeVLM/feature_dataset_all_labeled.csv"
```

The original `feature_dataset_all.csv` is untouched, and `thresholds.json` records exactly
which cut produced these labels — regenerate rather than guess if you change it later.

## Reading the result

| Sign | What it means |
|---|---|
| `REFIXATION` is healthy (>5%) | the two-threshold design has real cases to justify it |
| `REFIXATION` is tiny | either lower `FRAME_PCT` / raise `GAZE_PCT`, or accept that the claim rests on few rows |
| `PURSUIT` is tiny | expected — the 7×7 grid mixes background into the gaze patch. Raising `dino_input_patch` to 224 would recover it |
| ICC > 0.4 | the label is partly scene identity; consider `PER_VIDEO = True` |
| per-split balances differ a lot | reshuffle the split, or stratify on `quad` instead of mean similarity |

## One thing this notebook does not do

It labels rows using the **true** DINOv2 similarities — this is the *oracle*, what a
perfect predictor would decide. The actual gate has to predict these from gaze alone.
So treat `quad` as the target, never as a feature.